# Private Property Transactions: Modelling Setup
Initial data cleaning, feature engineering, and dataset splitting to support hypothesis testing and future price prediction models.

In [15]:
# Imports and notebook configuration
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import train_test_split

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

DATA_DIR = Path("data")
data_path = DATA_DIR / "cleaned_data_geocoded.csv"

In [16]:
# Load the combined private transactions dataset
raw_df = pd.read_csv(data_path)
print(f"Loaded {raw_df.shape[0]:,} rows and {raw_df.shape[1]} columns from {data_path.name}.")
raw_df.head()

Loaded 139,317 rows and 23 columns from cleaned_data_geocoded.csv.


,Project Name,Transacted Price ($),Area (SQFT),Unit Price ($ PSF),Sale Date,Street Name,Type of Sale,Type of Area,Area (SQM),Unit Price ($ PSM),Nett Price($),Property Type,Number of Units,Tenure,Postal District,Market Segment,Floor Level,Sale Year,Sale Month,Sale Quarter,Property Type Grouped,Latitude,Longitude
0,HARBOUR RESIDENCES,6340000,2185.09,2901,2025-09-01,PASIR PANJANG ROAD,New Sale,Land,203.0,31232,NaN,Terrace House,1,Freehold,5,Rest of Central Region,-,2025,9,2025Q3,Terrace House,1.281188,103.784414
1,LANDED HOUSING DEVELOPMENT,15200000,7812.51,1946,2025-09-01,OCEAN DRIVE,Resale,Land,725.8,20942,NaN,Detached House,1,99 yrs lease commencing from 2004,4,Core Central Region,-,2025,9,2025Q3,Detached House,1.249586,103.841621
2,BLAIR PLAIN CONSERVATION AREA,5900000,2037.63,2896,2025-09-01,BLAIR ROAD,Resale,Land,189.3,31167,NaN,Terrace House,1,Freehold,2,Rest of Central Region,-,2025,9,2025Q3,Terrace House,1.276113,103.836121
3,HARBOUR RESIDENCES,5630000,1983.81,2838,2025-09-01,PASIR PANJANG ROAD,New Sale,Land,184.3,30548,NaN,Terrace House,1,Freehold,5,Rest of Central Region,-,2025,9,2025Q3,Terrace House,1.281188,103.784414
4,HARBOUR RESIDENCES,5928000,2046.24,2897,2025-09-01,PASIR PANJANG ROAD,New Sale,Land,190.1,31184,NaN,Terrace House,1,Freehold,5,Rest of Central Region,-,2025,9,2025Q3,Terrace House,1.281188,103.784414


In [17]:
raw_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 139317 entries, 0 to 139316
Data columns (total 23 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   Project Name           139317 non-null  object 
 1   Transacted Price ($)   139317 non-null  int64  
 2   Area (SQFT)            139317 non-null  float64
 3   Unit Price ($ PSF)     139317 non-null  int64  
 4   Sale Date              139317 non-null  object 
 5   Street Name            139317 non-null  object 
 6   Type of Sale           139317 non-null  object 
 7   Type of Area           139317 non-null  object 
 8   Area (SQM)             139317 non-null  float64
 9   Unit Price ($ PSM)     139317 non-null  int64  
 10  Nett Price($)          246 non-null     float64
 11  Property Type          139317 non-null  object 
 12  Number of Units        139317 non-null  int64  
 13  Tenure                 139317 non-null  object 
 14  Postal District        139317 non-nu

In [18]:
raw_df = pd.read_csv(data_path)
raw_df = raw_df.replace(r"^\s*-\s*$", pd.NA, regex=True)

In [19]:
# Clean numeric columns and convert to numeric dtypes
clean_df = raw_df.copy()

numeric_columns = [
    "Transacted Price ($)",
    "Area (SQFT)",
    "Unit Price ($ PSF)",
    "Area (SQM)",
    "Unit Price ($ PSM)",
    "Nett Price($)",
    "Number of Units",
]

def parse_numeric(series: pd.Series) -> pd.Series:
    cleaned = series.astype(str).str.strip()
    cleaned = cleaned.str.replace(r"[,$()]", "", regex=True)
    cleaned = cleaned.replace({r"^\s*-+$": np.nan, r"(?i)^nan$": np.nan, "": np.nan}, regex=True)
    return pd.to_numeric(cleaned, errors="coerce")

for col in numeric_columns:
    if col in clean_df.columns:
        clean_df[col] = parse_numeric(clean_df[col])

clean_df[numeric_columns].describe().T

,count,mean,std,min,25%,50%,75%,max
Transacted Price ($),139317.0,2.139910e+06,4.620993e+06,320000.00,1228888.00,1623800.00,2320000.00,8.900000e+08
Area (SQFT),139317.0,1.258353e+03,2.421592e+03,258.34,731.95,1011.82,1323.97,6.198342e+05
Unit Price ($ PSF),139317.0,1.767399e+03,6.106595e+02,120.00,1304.00,1680.00,2167.00,6.593000e+03
Area (SQM),139317.0,1.169038e+02,2.249714e+02,24.00,68.00,94.00,123.00,5.758400e+04
Unit Price ($ PSM),139317.0,1.902429e+04,6.573142e+03,1288.00,14035.00,18089.00,23327.00,7.096400e+04
Nett Price($),246.0,2.304599e+06,2.210020e+06,802286.00,1272032.00,1799300.00,2330300.00,1.446300e+07
Number of Units,139317.0,1.010207e+00,1.394298e+00,1.00,1.00,1.00,1.00,4.460000e+02


In [20]:
# Convert selected columns to categorical dtypes for modelling convenience
categorical_columns = [
    "Property Type",
    "Type of Sale",
    "Type of Area",
    "Postal District",
    "Market Segment",
]

for col in categorical_columns:
    if col in clean_df.columns:
        clean_df[col] = clean_df[col].astype("string").str.strip()
        if col == "Postal District":
            clean_df[col] = clean_df[col].str.zfill(2)
        clean_df[col] = clean_df[col].astype("category")

clean_df[categorical_columns].dtypes

Property Type      category
Type of Sale       category
Type of Area       category
Postal District    category
Market Segment     category
dtype: object

In [21]:
clean_df.head()

,Project Name,Transacted Price ($),Area (SQFT),Unit Price ($ PSF),Sale Date,Street Name,Type of Sale,Type of Area,Area (SQM),Unit Price ($ PSM),Nett Price($),Property Type,Number of Units,Tenure,Postal District,Market Segment,Floor Level,Sale Year,Sale Month,Sale Quarter,Property Type Grouped,Latitude,Longitude
0,HARBOUR RESIDENCES,6340000,2185.09,2901,2025-09-01,PASIR PANJANG ROAD,New Sale,Land,203.0,31232,NaN,Terrace House,1,Freehold,05,Rest of Central Region,<NA>,2025,9,2025Q3,Terrace House,1.281188,103.784414
1,LANDED HOUSING DEVELOPMENT,15200000,7812.51,1946,2025-09-01,OCEAN DRIVE,Resale,Land,725.8,20942,NaN,Detached House,1,99 yrs lease commencing from 2004,04,Core Central Region,<NA>,2025,9,2025Q3,Detached House,1.249586,103.841621
2,BLAIR PLAIN CONSERVATION AREA,5900000,2037.63,2896,2025-09-01,BLAIR ROAD,Resale,Land,189.3,31167,NaN,Terrace House,1,Freehold,02,Rest of Central Region,<NA>,2025,9,2025Q3,Terrace House,1.276113,103.836121
3,HARBOUR RESIDENCES,5630000,1983.81,2838,2025-09-01,PASIR PANJANG ROAD,New Sale,Land,184.3,30548,NaN,Terrace House,1,Freehold,05,Rest of Central Region,<NA>,2025,9,2025Q3,Terrace House,1.281188,103.784414
4,HARBOUR RESIDENCES,5928000,2046.24,2897,2025-09-01,PASIR PANJANG ROAD,New Sale,Land,190.1,31184,NaN,Terrace House,1,Freehold,05,Rest of Central Region,<NA>,2025,9,2025Q3,Terrace House,1.281188,103.784414


In [24]:
drop_columns = ["Nett Price($)", "Tenure", "tenure"]
clean_df = clean_df.drop(columns=[c for c in drop_columns if c in clean_df.columns]) # type: ignore
clean_df.head()

,Project Name,Transacted Price ($),Area (SQFT),Unit Price ($ PSF),Sale Date,Street Name,Type of Sale,Type of Area,Area (SQM),Unit Price ($ PSM),Property Type,Number of Units,Postal District,Market Segment,Floor Level,Sale Year,Sale Month,Sale Quarter,Property Type Grouped,Latitude,Longitude
0,HARBOUR RESIDENCES,6340000,2185.09,2901,2025-09-01,PASIR PANJANG ROAD,New Sale,Land,203.0,31232,Terrace House,1,05,Rest of Central Region,<NA>,2025,9,2025Q3,Terrace House,1.281188,103.784414
1,LANDED HOUSING DEVELOPMENT,15200000,7812.51,1946,2025-09-01,OCEAN DRIVE,Resale,Land,725.8,20942,Detached House,1,04,Core Central Region,<NA>,2025,9,2025Q3,Detached House,1.249586,103.841621
2,BLAIR PLAIN CONSERVATION AREA,5900000,2037.63,2896,2025-09-01,BLAIR ROAD,Resale,Land,189.3,31167,Terrace House,1,02,Rest of Central Region,<NA>,2025,9,2025Q3,Terrace House,1.276113,103.836121
3,HARBOUR RESIDENCES,5630000,1983.81,2838,2025-09-01,PASIR PANJANG ROAD,New Sale,Land,184.3,30548,Terrace House,1,05,Rest of Central Region,<NA>,2025,9,2025Q3,Terrace House,1.281188,103.784414
4,HARBOUR RESIDENCES,5928000,2046.24,2897,2025-09-01,PASIR PANJANG ROAD,New Sale,Land,190.1,31184,Terrace House,1,05,Rest of Central Region,<NA>,2025,9,2025Q3,Terrace House,1.281188,103.784414


In [ ]:
# Helper to load GeoJSON points and compute nearest distance features
import json
from functools import lru_cache
from typing import Iterable, Tuple

geo_features_dir = Path("geo-features")

@lru_cache(maxsize=None)
def load_geojson_points(filename: str) -> tuple[Tuple[float, float], ...]:
    geo_path = geo_features_dir / filename
    if not geo_path.exists():
        raise FileNotFoundError(f"Expected geojson dataset at {geo_path}")
    with geo_path.open() as f:
        geo_data = json.load(f)

    coords: list[Tuple[float, float]] = []
    for feature in geo_data.get("features", []):
        geometry = feature.get("geometry", {})
        if geometry.get("type") != "Point":
            continue
        raw_coords = geometry.get("coordinates")
        if not raw_coords or len(raw_coords) < 2:
            continue
        lon, lat = raw_coords[:2]
        if lat is None or lon is None:
            continue
        coords.append((float(lat), float(lon)))
    if not coords:
        raise ValueError(f"No point coordinates found in {filename}")
    return tuple(coords)

def compute_nearest_distance(lat_series: pd.Series, lon_series: pd.Series, reference_points: Iterable[Tuple[float, float]]) -> np.ndarray:
    points_array = np.array(list(reference_points), dtype=float)
    ref_lat_rad = np.radians(points_array[:, 0])
    ref_lon_rad = np.radians(points_array[:, 1])

    valid_mask = lat_series.notna() & lon_series.notna()
    distances = np.full(lat_series.shape[0], np.nan, dtype=float)
    if not valid_mask.any():
        warnings.warn("No rows with both latitude and longitude available for distance calculation.", stacklevel=2)
        return distances

    lat_rad = np.radians(lat_series[valid_mask].to_numpy())
    lon_rad = np.radians(lon_series[valid_mask].to_numpy())
    dlat = lat_rad[:, None] - ref_lat_rad[None, :]
    dlon = lon_rad[:, None] - ref_lon_rad[None, :]
    hav_a = np.sin(dlat / 2.0) ** 2 + np.cos(lat_rad)[:, None] * np.cos(ref_lat_rad)[None, :] * np.sin(dlon / 2.0) ** 2
    earth_radius_km = 6371.0
    min_distances = earth_radius_km * (2.0 * np.arcsin(np.sqrt(hav_a)))
    distances[valid_mask.to_numpy()] = np.min(min_distances, axis=1)
    missing_coords = (~valid_mask).sum()
    if missing_coords:
        warnings.warn(f"Skipping distance computation for {missing_coords} rows without latitude/longitude.", stacklevel=2)
    return distances

In [29]:
# Generate distance-to-feature columns for selected GeoJSON files
distance_jobs = {
    "GymsSGGEOJSON.geojson": "distToGym",
    "HawkerCentresGEOJSON.geojson": "distToHawker",
    "SupermarketsGEOJSON.geojson": "distToSupermarket",
    "ParksSG.geojson": "distToPark",
    "HistoricSitesGEOJSON.geojson": "distToHistoricSite",
    "TouristAttractions.geojson": "distToTourist",
    "LTAMRTStationExitGEOJSON.geojson": "distToMrtExit",
    "LTATaxiStopGEOJSON.geojson": "distToTaxiStop",
    "DisabilityServices.geojson": "distToDisabilitySvc",
    "MonumentsGEOJSON.geojson": "distToMonument",
}

for filename, column_name in distance_jobs.items():
    reference_points = load_geojson_points(filename)
    clean_df[column_name] = compute_nearest_distance(clean_df["Latitude"], clean_df["Longitude"], reference_points)
    print(f"Added {column_name} from {filename} with {len(reference_points)} reference points.")

clean_df[["Latitude", "Longitude"] + list(distance_jobs.values())].head()

/var/folders/nt/67mjll3s4fb1vnp8mvn0q2kh0000gn/T/ipykernel_61248/3579326213.py:17: UserWarning: Skipping distance computation for 609 rows without latitude/longitude.
  clean_df[column_name] = compute_nearest_distance(clean_df["Latitude"], clean_df["Longitude"], reference_points)


Added distToGym from GymsSGGEOJSON.geojson with 159 reference points.
Added distToHawker from HawkerCentresGEOJSON.geojson with 129 reference points.
Added distToHawker from HawkerCentresGEOJSON.geojson with 129 reference points.


/var/folders/nt/67mjll3s4fb1vnp8mvn0q2kh0000gn/T/ipykernel_61248/3579326213.py:17: UserWarning: Skipping distance computation for 609 rows without latitude/longitude.
  clean_df[column_name] = compute_nearest_distance(clean_df["Latitude"], clean_df["Longitude"], reference_points)
/var/folders/nt/67mjll3s4fb1vnp8mvn0q2kh0000gn/T/ipykernel_61248/3579326213.py:17: UserWarning: Skipping distance computation for 609 rows without latitude/longitude.
  clean_df[column_name] = compute_nearest_distance(clean_df["Latitude"], clean_df["Longitude"], reference_points)


Added distToSupermarket from SupermarketsGEOJSON.geojson with 526 reference points.
Added distToPark from ParksSG.geojson with 52 reference points.


/var/folders/nt/67mjll3s4fb1vnp8mvn0q2kh0000gn/T/ipykernel_61248/3579326213.py:17: UserWarning: Skipping distance computation for 609 rows without latitude/longitude.
  clean_df[column_name] = compute_nearest_distance(clean_df["Latitude"], clean_df["Longitude"], reference_points)


...

Added distToMrtExit from LTAMRTStationExitGEOJSON.geojson with 563 reference points.


/var/folders/nt/67mjll3s4fb1vnp8mvn0q2kh0000gn/T/ipykernel_61248/3579326213.py:17: UserWarning: Skipping distance computation for 609 rows without latitude/longitude.
  clean_df[column_name] = compute_nearest_distance(clean_df["Latitude"], clean_df["Longitude"], reference_points)


Added distToTaxiStop from LTATaxiStopGEOJSON.geojson with 366 reference points.
Added distToDisabilitySvc from DisabilityServices.geojson with 154 reference points.
Added distToMonument from MonumentsGEOJSON.geojson with 73 reference points.
Added distToDisabilitySvc from DisabilityServices.geojson with 154 reference points.
Added distToMonument from MonumentsGEOJSON.geojson with 73 reference points.


/var/folders/nt/67mjll3s4fb1vnp8mvn0q2kh0000gn/T/ipykernel_61248/3579326213.py:17: UserWarning: Skipping distance computation for 609 rows without latitude/longitude.
  clean_df[column_name] = compute_nearest_distance(clean_df["Latitude"], clean_df["Longitude"], reference_points)
/var/folders/nt/67mjll3s4fb1vnp8mvn0q2kh0000gn/T/ipykernel_61248/3579326213.py:17: UserWarning: Skipping distance computation for 609 rows without latitude/longitude.
  clean_df[column_name] = compute_nearest_distance(clean_df["Latitude"], clean_df["Longitude"], reference_points)


,Latitude,Longitude,distToGym,distToHawker,distToSupermarket,distToPark,distToHistoricSite,distToTourist,distToMrtExit,distToTaxiStop,distToDisabilitySvc,distToMonument
0,1.281188,103.784414,1.775594,0.973825,1.288904,0.752742,0.759446,0.553540,0.338780,0.386634,2.672215,4.306728
1,1.249586,103.841621,2.750026,3.010256,0.205876,3.031460,0.586978,1.505636,2.735700,2.664471,3.535999,2.509141
2,1.276113,103.836121,0.555165,0.784665,0.532839,1.030570,0.413142,0.164900,0.422934,0.322589,0.531525,0.378783
3,1.281188,103.784414,1.775594,0.973825,1.288904,0.752742,0.759446,0.553540,0.338780,0.386634,2.672215,4.306728
4,1.281188,103.784414,1.775594,0.973825,1.288904,0.752742,0.759446,0.553540,0.338780,0.386634,2.672215,4.306728


In [30]:
clean_df

,Project Name,Transacted Price ($),Area (SQFT),Unit Price ($ PSF),Sale Date,Street Name,Type of Sale,Type of Area,Area (SQM),Unit Price ($ PSM),Property Type,Number of Units,Postal District,Market Segment,Floor Level,Sale Year,Sale Month,Sale Quarter,Property Type Grouped,Latitude,Longitude,distToGym,distToHawker,distToSupermarket,distToPark,distToHistoricSite,distToTourist,distToMrtExit,distToTaxiStop,distToDisabilitySvc,distToMonument
0,HARBOUR RESIDENCES,6340000,2185.09,2901,2025-09-01,PASIR PANJANG ROAD,New Sale,Land,203.0,31232,Terrace House,1,05,Rest of Central Region,<NA>,2025,9,2025Q3,Terrace House,1.281188,103.784414,1.775594,0.973825,1.288904,0.752742,0.759446,0.553540,0.338780,0.386634,2.672215,4.306728
1,LANDED HOUSING DEVELOPMENT,15200000,7812.51,1946,2025-09-01,OCEAN DRIVE,Resale,Land,725.8,20942,Detached House,1,04,Core Central Region,<NA>,2025,9,2025Q3,Detached House,1.249586,103.841621,2.750026,3.010256,0.205876,3.031460,0.586978,1.505636,2.735700,2.664471,3.535999,2.509141
2,BLAIR PLAIN CONSERVATION AREA,5900000,2037.63,2896,2025-09-01,BLAIR ROAD,Resale,Land,189.3,31167,Terrace House,1,02,Rest of Central Region,<NA>,2025,9,2025Q3,Terrace House,1.276113,103.836121,0.555165,0.784665,0.532839,1.030570,0.413142,0.164900,0.422934,0.322589,0.531525,0.378783
3,HARBOUR RESIDENCES,5630000,1983.81,2838,2025-09-01,PASIR PANJANG ROAD,New Sale,Land,184.3,30548,Terrace House,1,05,Rest of Central Region,<NA>,2025,9,2025Q3,Terrace House,1.281188,103.784414,1.775594,0.973825,1.288904,0.752742,0.759446,0.553540,0.338780,0.386634,2.672215,4.306728
4,HARBOUR RESIDENCES,5928000,2046.24,2897,2025-09-01,PASIR PANJANG ROAD,New Sale,Land,190.1,31184,Terrace House,1,05,Rest of Central Region,<NA>,2025,9,2025Q3,Terrace House,1.281188,103.784414,1.775594,0.973825,1.288904,0.752742,0.759446,0.553540,0.338780,0.386634,2.672215,4.306728
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
139312,RV ALTITUDE,1750000,624.31,2803,2020-09-01,RIVER VALLEY ROAD,New Sale,Strata,58.0,30172,Apartment,1,09,Core Central Region,06 to 10,2020,9,2020Q3,Apartment,1.293061,103.826480,0.051409,0.480303,0.051409,0.653247,0.031381,1.281243,0.683641,0.153978,0.358967,1.585930
139313,FOURTH AVENUE RESIDENCES,1115000,484.38,2302,2020-09-01,FOURTH AVENUE,New Sale,Strata,45.0,24778,Apartment,1,10,Core Central Region,01 to 05,2020,9,2020Q3,Apartment,1.328924,103.797052,0.249111,1.702930,0.259116,2.602289,1.357971,2.674089,0.213337,0.234923,3.515660,0.645167
139314,KOPAR AT NEWTON,1537000,688.90,2231,2020-09-01,MAKEWAY AVENUE,New Sale,Strata,64.0,24016,Apartment,1,09,Core Central Region,06 to 10,2020,9,2020Q3,Apartment,1.312542,103.841921,0.291031,0.270445,0.559853,1.502139,0.989908,1.072792,0.414727,0.468104,1.710915,0.620712
139315,JERVOIS LODGE,1750000,1388.56,1260,2020-09-01,JERVOIS ROAD,Resale,Strata,129.0,13566,Condominium,1,10,Core Central Region,01 to 05,2020,9,2020Q3,Condominium,1.294943,103.821243,0.668604,0.852387,0.668604,0.887158,0.585461,1.602738,0.751887,0.466609,0.528869,1.288573


In [31]:
clean_df.to_csv("data/cleaned_with_features.csv", index=False)